In [2]:
import tensorflow as tf
import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Dense, Flatten, MaxPooling2D, GlobalAveragePooling2D, Dropout
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.model_selection import train_test_split
import numpy as np
import os

In [41]:
dataset_path = "./dataset"
classes = os.listdir(dataset_path)
image_width = 224
image_height = 224

x = []
y = []
label = 0

In [42]:
for class_name in classes:
    class_path = os.path.join(dataset_path, class_name)
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        img = cv2.imread(img_path)

        if img is None:
            continue

        img = cv2.resize(img, (image_width, image_height))
        img = img / 255.0  # NORMALISASI
        x.append(img)
        y.append(label)
    label += 1

x = np.array(x)
y = np.array(y)

print("Jumlah data:", len(x))
print("Jumlah kelas:", len(classes))

Jumlah data: 925
Jumlah kelas: 8


In [43]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, shuffle=True
)

y_train = tf.keras.utils.to_categorical(y_train, num_classes = 8)
y_test = tf.keras.utils.to_categorical(y_test, num_classes = 8)

In [44]:
data_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
])

base = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)
base.trainable = False  # freeze awal

model = Sequential([
    data_aug,
    base,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.4),
    Dense(8, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_5 (Sequential)       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_7      │ ?                      │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,257,984 (8.61 MB)

In [45]:
history = model.fit(
    x_train, y_train,
    epochs=20,
    batch_size=32,
)


Epoch 1/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 13s 335ms/step - accuracy: 0.3766 - loss: 1.7966
Epoch 2/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 337ms/step - accuracy: 0.7399 - loss: 0.7312
Epoch 3/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 319ms/step - accuracy: 0.8178 - loss: 0.5468
Epoch 4/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 315ms/step - accuracy: 0.8862 - loss: 0.3613
Epoch 5/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 309ms/step - accuracy: 0.8934 - loss: 0.3449
Epoch 6/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 310ms/step - accuracy: 0.8955 - loss: 0.3127
Epoch 7/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 320ms/step - accuracy: 0.9403 - loss: 0.2104
Epoch 8/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 313ms/step - accuracy: 0.9343 - loss: 0.2394
Epoch 9/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 7s 309ms/step - accuracy: 0.9491 - loss: 0.1705
Epoch 10/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 7s 297ms/step - accuracy: 0.9489 - loss: 0.1567
Epoch 11/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 315ms/step - accuracy: 0.9455 - loss: 0.1827
Epoch 12/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 7s 298ms/ste

In [ ]:
loss, acc = model.evaluate(x_test, y_test)
print("Final Accuracy:", acc)
print("Final Loss:", loss)
 

6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 261ms/step - accuracy: 0.9220 - loss: 0.2579
Final Accuracy: 0.9405405521392822
Final Loss: 0.21631774306297302


In [47]:
model.save("my_model.h5")
print("Model saved as my_model.h5")

Model saved as my_model.h5


In [3]:
print(tf.__version__)

2.19.0
